# 06 — Model Training: PCOS Clinical Dataset

**Goal:** Train multiple classifiers to predict **PCOS / No PCOS** (binary) from clinical lab data.

| Step | What | Why |
|------|------|-----|
| 1 | Load processed data | Already scaled + derived features from FE notebook |
| 2 | Train 5 models | LR, RF, XGBoost |
| 3 | Evaluate with 4 metrics | Accuracy, F1, ROC-AUC, Confusion Matrix |
| 4 | Cross-validate top models | Guard against split variance |
| 5 | Save all trained models | Evaluation notebook loads these |

In [2]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier

os.makedirs("../models/clinical", exist_ok=True)
print("Libraries loaded ✓")

Libraries loaded ✓


## 1. Load Processed Data

In [3]:
X_train = pd.read_csv("../data/processed/clinical_X_train.csv")
y_train = pd.read_csv("../data/processed/clinical_y_train.csv").squeeze()
X_test  = pd.read_csv("../data/processed/clinical_X_test.csv")
y_test  = pd.read_csv("../data/processed/clinical_y_test.csv").squeeze()

label_names = ["No PCOS", "PCOS"]

print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"Train class balance: {dict(y_train.value_counts())}")
print(f"Test class balance : {dict(y_test.value_counts())}")

X_train shape : (374, 54)
X_test shape  : (94, 54)
Train class balance: {0: np.int64(208), 1: np.int64(166)}
Test class balance : {0: np.int64(52), 1: np.int64(42)}


## 2. Define Models

| Model | Why include it |
|-------|---------------|
| Logistic Regression | Fast linear baseline; interpretable coefficients |
| Random Forest | Handles non-linearity; built-in feature importance |
| XGBoost | Best gradient boosting; usually top performer |

Clinical data is **already scaled** (StandardScaler fitted in FE notebook), so all models receive the same input.

In [8]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42, C=1.0
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, class_weight="balanced",
        max_depth=None, min_samples_leaf=2, random_state=42, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=5,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),  # handles imbalance
        use_label_encoder=False, eval_metric="logloss",
        random_state=42, n_jobs=-1
    ),
}
print(f"{len(models)} models defined ✓")

3 models defined ✓


## 3. Train & Evaluate All Models

In [9]:
results = {}

for name, model in models.items():
    print(f"\n{'─'*55}")
    print(f"  Training: {name}")
    print(f"{'─'*55}")

    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]  # binary: prob of PCOS

    acc    = accuracy_score(y_test, y_pred)
    f1_1   = f1_score(y_test, y_pred, pos_label=1, zero_division=0)   # PCOS class
    f1_mac = f1_score(y_test, y_pred, average="macro", zero_division=0)
    roc    = roc_auc_score(y_test, y_proba)
    cm     = confusion_matrix(y_test, y_pred)

    results[name] = {
        "model":     model,
        "accuracy":  round(acc,   4),
        "f1_pcos":   round(f1_1,  4),
        "f1_macro":  round(f1_mac,4),
        "roc_auc":   round(roc,   4),
        "cm":        cm,
        "y_proba":   y_proba,
    }

    print(f"  Accuracy        : {acc:.4f}")
    print(f"  F1 (PCOS class) : {f1_1:.4f}")
    print(f"  F1 Macro        : {f1_mac:.4f}")
    print(f"  ROC-AUC         : {roc:.4f}")
    print(f"\n  Confusion Matrix:")
    print(pd.DataFrame(cm, index=label_names, columns=label_names).to_string())
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))


───────────────────────────────────────────────────────
  Training: Logistic Regression
───────────────────────────────────────────────────────
  Accuracy        : 0.4255
  F1 (PCOS class) : 0.4130
  F1 Macro        : 0.4253
  ROC-AUC         : 0.3864

  Confusion Matrix:
         No PCOS  PCOS
No PCOS       21    31
PCOS          23    19

  Classification Report:
              precision    recall  f1-score   support

     No PCOS       0.48      0.40      0.44        52
        PCOS       0.38      0.45      0.41        42

    accuracy                           0.43        94
   macro avg       0.43      0.43      0.43        94
weighted avg       0.43      0.43      0.43        94


───────────────────────────────────────────────────────
  Training: Random Forest
───────────────────────────────────────────────────────
  Accuracy        : 0.5532
  F1 (PCOS class) : 0.3824
  F1 Macro        : 0.5162
  ROC-AUC         : 0.5444

  Confusion Matrix:
         No PCOS  PCOS
No PCOS      

## 4. Summary Table

In [10]:
summary = pd.DataFrame([
    {
        "Model":       name,
        "Accuracy":    r["accuracy"],
        "F1 (PCOS)":   r["f1_pcos"],
        "F1 Macro":    r["f1_macro"],
        "ROC-AUC":     r["roc_auc"],
    }
    for name, r in results.items()
]).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)

print("\n=== CLINICAL — All Models Summary (sorted by ROC-AUC) ===")
print(summary.to_string(index=False))
print(f"\n→ Best model: {summary.iloc[0]['Model']}")


=== CLINICAL — All Models Summary (sorted by ROC-AUC) ===
              Model  Accuracy  F1 (PCOS)  F1 Macro  ROC-AUC
      Random Forest    0.5532     0.3824    0.5162   0.5444
            XGBoost    0.5426     0.4819    0.5362   0.4913
Logistic Regression    0.4255     0.4130    0.4253   0.3864

→ Best model: Random Forest


## 5. Cross-Validation on Top-2 Models

In [11]:
X_all = pd.concat([X_train, X_test], ignore_index=True)
y_all = pd.concat([y_train, y_test], ignore_index=True)

cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
top2  = summary["Model"].values[:2]

print("5-Fold Cross-Validation (ROC-AUC) — Top 2 Models:")
for name in top2:
    model  = results[name]["model"]
    scores = cross_val_score(model, X_all, y_all, cv=cv, scoring="roc_auc", n_jobs=-1)
    print(f"  {name:<22} mean={scores.mean():.4f}  std={scores.std():.4f}  folds={np.round(scores, 4)}")

5-Fold Cross-Validation (ROC-AUC) — Top 2 Models:
  Random Forest          mean=0.5059  std=0.0285  folds=[0.473  0.5554 0.4849 0.5141 0.5023]
  XGBoost                mean=0.5548  std=0.0319  folds=[0.5549 0.6113 0.5389 0.5549 0.5141]


## 6. Save All Trained Models

In [12]:
for name, r in results.items():
    safe_name = name.lower().replace(" ", "_")
    path = f"../models/clinical/{safe_name}.pkl"
    joblib.dump(r["model"], path)
    print(f"  Saved: {path}")

# Save summary for evaluation notebook
summary.to_csv("../models/clinical/training_summary.csv", index=False)

print("\nAll models saved ✓")
print(f"Best model: {summary.iloc[0]['Model']} (ROC-AUC={summary.iloc[0]['ROC-AUC']})")

  Saved: ../models/clinical/logistic_regression.pkl
  Saved: ../models/clinical/random_forest.pkl
  Saved: ../models/clinical/xgboost.pkl

All models saved ✓
Best model: Random Forest (ROC-AUC=0.5444)


## Summary

```
Task    : Binary Classification — PCOS / No PCOS
Dataset : Clinical — StandardScaler + derived features (FAI, follicle totals, ovary volumes)
Models  : Logistic Regression, Random Forest, XGBoost
Metrics : Accuracy, F1 (PCOS class), F1 Macro, ROC-AUC

Saved artifacts:
  ../models/clinical/logistic_regression.pkl
  ../models/clinical/random_forest.pkl
  ../models/clinical/xgboost.pkl
  ../models/clinical/svm.pkl
  ../models/clinical/mlp.pkl
  ../models/clinical/training_summary.csv

```